# C4-classical-ml-practice — Session 1: Pandas and Data Loading

*One class session, roughly 80 minutes. Prerequisites: C1-ml-fundamentals
(supervised classification, train/test discipline, accuracy),
F1-scientific-python (arrays, indexing, broadcasting, axis aggregations,
seeded randomness), and — for later sessions — F2-vectors (distances) and
F5-probability (variance).*

**This session:** real datasets arrive as *tables* in CSV files, not as
ready-made NumPy arrays.
This session teaches the table tool — pandas — from zero: what a
DataFrame is, the CSV round trip, selecting columns and rows, boolean
filtering, group summaries, and the single most exam-relevant move in
the whole library: the **bridge from a labeled table to the `(X, y)`
NumPy pair** that every classifier eats.
It closes with a fully worked exam-style loading task and the pitfalls
that cost real points.

Try every checkpoint yourself before running the verification cells.
Answers are collected at the end of this notebook.

In [ ]:
import numpy as np
import pandas as pd

## 1. From Arrays to Tables

**Motivation.**
In C1 you classified data that was already a clean numeric array.
Real projects start earlier: a file where each row is one measured
*thing* (a bean, a machine, a patient) and each column is one named
attribute — some numeric, some text.
NumPy has no column names and only one dtype per array, so mixed tables
need a second tool: **pandas**.
pandas is the loading dock; NumPy stays the workshop.

**Definition.**
A **DataFrame** is a 2-D table with

- **named columns**, each a **Series** (a 1-D labeled array) with its
  own dtype — one column can hold strings while its neighbor holds
  floats;
- an **index** of row labels (by default the integers 0, 1, 2, …);
- a `shape` attribute `(n_rows, n_columns)`, exactly like an array.

The cleanest constructor for small tables is a dict:
keys become column names, values become columns.

**Worked example.**
Five beans, one label column and two measurements:

In [ ]:
mini = pd.DataFrame({
    "species":   ["alba", "cava", "alba", "brio", "cava"],
    "length_mm": [11.2, 15.8, 10.9, 13.1, 16.0],
    "mass_g":    [0.42, 0.85, 0.39, 0.61, 0.79],
})
print(mini)
print()
print("shape :", mini.shape)          # (5 rows, 3 columns)
print("columns:", list(mini.columns))
print()
print(mini.dtypes)                    # per-COLUMN dtypes -- the point of pandas

The `species` column has dtype `object`/`str` (text), the measurements
are `float64` — three columns, two kinds of data, one table.
An ndarray could not store this without forcing everything to a common
type.

### Checkpoint 1

1. Build a DataFrame `pets` with columns `name` (three strings) and
   `weight_kg` (three floats).
   What are its `shape` and its two dtypes?
2. In one sentence: why can a DataFrame hold a text column next to a
   float column when a single NumPy array cannot?
3. What does `mini["mass_g"]` return — a DataFrame, a Series, or an
   ndarray?
   Check with `type(...)`.

## 2. The CSV Round Trip

**Motivation.**
Tables travel between programs as **CSV** ("comma-separated values")
files: a plain-text format with one header line naming the columns,
then one line per row.
Every dataset in this unit — and the data file in the exam's applied
problem — arrives this way.

**The two verbs.**

- `df.to_csv(path, index=False)` writes a DataFrame to disk.
  `index=False` keeps the row labels *out* of the file — without it a
  stray unnamed column shows up on reload.
- `pd.read_csv(path)` reads a CSV into a DataFrame, inferring each
  column's dtype from its contents.

**First-look tools** — run these on *every* freshly loaded file, before
any analysis:

| Tool | Question it answers |
|---|---|
| `df.shape` | how many rows and columns? |
| `df.head(n)` | what do the first rows actually look like? |
| `df.dtypes` | did numbers load as numbers? |
| `df.describe()` | ranges/means of the numeric columns — anything absurd? |
| `df["label_col"].value_counts()` | how many rows per class? (Section 3) |

**Worked example.**
Write the mini table to a temporary file, look at the raw text, read it
back:

In [ ]:
import tempfile
from pathlib import Path

tmp = Path(tempfile.mkdtemp())          # scratch directory, not the repo
csv_path = tmp / "mini_beans.csv"
mini.to_csv(csv_path, index=False)

print(csv_path.read_text())             # the file is just text: header + rows

reloaded = pd.read_csv(csv_path)
print(reloaded)
print()
print(reloaded.dtypes)                  # dtypes inferred again on load

In [ ]:
# describe(): instant sanity summary of the numeric columns.
print(reloaded.describe().round(3))

### Checkpoint 2

1. Write `mini` to a CSV *without* `index=False`, read it back, and
   inspect the columns.
   What extra column appears, and where did it come from?
2. A colleague's CSV loads with `mass_g` as dtype `object` instead of
   `float64`.
   Name a plausible cause (think: what could sit in one cell of that
   column?) and the first-look tool that catches it.
3. Which single call tells you the number of rows, and which tells you
   whether any numeric column contains an absurd value like a negative
   mass?

## 3. Picking Columns

**Motivation.**
Analysis is mostly "take these columns, do something".
There are exactly two bracket forms, and their difference — Series
versus DataFrame — matters every time you hand data to NumPy or
sklearn.

**The two bracket forms.**

- `df["mass_g"]` — **single name → Series** (1-D).
  Series support arithmetic and aggregations like arrays:
  `df["mass_g"].mean()`, `df["mass_g"] * 1000`.
- `df[["length_mm", "mass_g"]]` — **list of names → DataFrame** (2-D),
  in exactly the order you listed.
  This is the form that becomes a feature *matrix*.

**Counting a text column.**
`df["species"].value_counts()` tabulates how many times each distinct
value occurs — the one-line class-balance check from C1.

In [ ]:
s = mini["mass_g"]                       # Series (1-D)
two = mini[["length_mm", "mass_g"]]      # DataFrame (2-D), column order as listed

print(type(s).__name__, s.shape, "| mean mass:", round(s.mean(), 3))
print(type(two).__name__, two.shape)
print()
print("mass in milligrams:")
print((s * 1000).head(3))                # arithmetic broadcasts over the column
print()
print(mini["species"].value_counts())    # class balance at a glance

### Checkpoint 3

1. From `mini`, get the *Series* of lengths and compute its max; then
   get the *DataFrame* holding `mass_g` and `length_mm` **in that
   order** and print its shape.
2. Predict the shapes: `mini["mass_g"].shape` versus
   `mini[["mass_g"]].shape`.
   Then verify.
3. Using one line, count how many beans of each species `mini`
   contains.

## 4. Picking Rows: Boolean Masks

**Motivation.**
"Only the heavy beans", "only species `cava`", "heavy **and** `cava`" —
row selection is filtering, and pandas filtering is the F1 boolean-mask
idea wearing column names.

**The pattern.**
A comparison on a Series produces a Series of `True`/`False` — a
**mask**; indexing the DataFrame with the mask keeps the `True` rows:

```python
df[df["mass_g"] > 0.6]
```

**Combining conditions.**
Use `&` (and), `|` (or), `~` (not) — *never* Python's `and`/`or`, which
cannot handle a whole column of booleans — **and wrap every comparison
in parentheses**: `&` binds tighter than `>`, so the unparenthesized
version is evaluated in the wrong order (Section 8 dissects the
failure).

**Counting with masks** (from F1): `mask.sum()` counts `True` rows;
`mask.mean()` is the fraction.

**Label-based selection with `.loc`.**
`df.loc[mask, ["length_mm", "mass_g"]]` filters rows *and* picks
columns in one step.
Its positional cousin `df.iloc[i]` fetches row(s) by integer position —
`mini.iloc[0]` is the first row regardless of index labels.

In [ ]:
heavy = mini[mini["mass_g"] > 0.6]
print(heavy)
print()

both = mini[(mini["mass_g"] > 0.6) & (mini["species"] == "cava")]   # parentheses!
print(both)
print()

print("n heavy:", (mini["mass_g"] > 0.6).sum(),
      "| fraction heavy:", (mini["mass_g"] > 0.6).mean())
print()
print(mini.loc[mini["species"] == "alba", ["length_mm", "mass_g"]])
print()
print("first row by position:")
print(mini.iloc[0])

Note the filtered rows *keep their original index labels* (0, 1, 4 …) —
filtering never renumbers.
`.reset_index(drop=True)` renumbers when you need a fresh 0..n-1 index.

### Checkpoint 4

1. Select the beans with `length_mm` below 12 **or** mass above 0.8
   (mind the parentheses), and count them with a mask aggregation.
2. What fraction of `mini`'s beans are species `cava`?
   One line.
3. Why does `mini[mini["mass_g"] > 0.6 & (mini["species"] == "cava")]`
   *not* compute "heavy and cava"?
   Say what gets evaluated first.

## 5. Group Summaries

**Motivation.**
"Mean mass *per species*" is one question per class — a loop in spirit.
`groupby` runs it loop-free: split the rows by a label column, apply an
aggregation to each group, return one row per group.

**The pattern.**

```python
df.groupby("species")["mass_g"].mean()
```

reads: *group rows by species → take each group's `mass_g` column →
average it*.
Swap `.mean()` for `.max()`, `.count()`, `.std()` … for other
summaries.
This is also a modeling reconnaissance tool: if the group means differ
a lot, that feature separates the classes.

In [ ]:
print(mini.groupby("species")["mass_g"].mean())
print()
print(mini.groupby("species")["length_mm"].max())
print()
print(mini.groupby("species").size())       # rows per group (same as value_counts)

### Checkpoint 5

1. Compute each species' mean `length_mm` in `mini`.
   Which species is longest on average?
2. In one sentence: for classification, why is a feature whose
   per-class group means are nearly identical a weak feature?
3. Write the `groupby` line that returns how many rows each species
   has, and name the Section 3 tool that answers the same question.

## 6. The NumPy Bridge: From Table to (X, y)

**Motivation.**
Classifiers do not eat DataFrames with named string columns; they eat a
numeric **feature matrix `X`** of shape `(n_samples, n_features)` and a
**label vector `y`** of shape `(n_samples,)` — the `(X, y)` pair from
C1.
Every applied task in this unit — and the real paper's applied problem,
whose starter code performs *exactly this move* (paraphrased) — begins:
load the CSV, split the table into feature columns and the label
column, convert to arrays.

**The idiom** (worth memorizing as a block):

```python
FEATURES = ["length_mm", "mass_g"]        # explicit list, fixed order
X = df[FEATURES].to_numpy()               # (n, d) float array
y = df["species"].to_numpy()              # (n,)   label array
```

Three deliberate choices:

- **An explicit feature list.**
  The column *order* in `X` is the order of the list; every later
  object (a scaler's per-column statistics, a query row you build by
  hand) silently assumes that same order.
- **`.to_numpy()`** hands back a plain ndarray — from here on,
  everything is F1/F2 territory.
- **The label column stays out of `X`.**
  Mixing it in does not crash — it does something *worse* (Section 8).

**Shape check ritual:** after the bridge, print `X.shape`, `X.dtype`,
`y.shape`.
Expected: `(n, d)` with a float dtype, and `(n,)`.

In [ ]:
FEATURES = ["length_mm", "mass_g"]

X = mini[FEATURES].to_numpy()
y = mini["species"].to_numpy()

print("X:", X.shape, X.dtype)
print(X)
print("y:", y.shape, y.dtype)     # object/str dtype is fine for LABELS
print(y)
print()
# X is a live NumPy array: F1 tools apply directly.
print("per-feature means:", X.mean(axis=0).round(3))

### Checkpoint 6

1. Build `X2` holding `mass_g` and `length_mm` **in that order** and
   `y2` from `species`.
   Verify the shapes, and state what `X2[:, 0]` measures (careful —
   the answer depends on the list order).
2. Why does the feature list `FEATURES` exist as a named variable
   instead of writing the column names inline twice?
   Answer with the failure it prevents.
3. Perform the ritual on `X2`, `y2`: shapes and dtypes.
   Which dtype must be numeric, and which may be text?

## 7. Worked Exam-Style Example: The Loading Task

The real Round 1 applied problem hands you a CSV and starter code in
exactly Section 6's idiom; its data-wrangling sub-parts are graded on
precise identifiers and shapes (paraphrased register — no verbatim test
text).
Here is a full task in that register, solved step by step, using this
unit's committed dataset `practice/data/beans.csv` (generated by the
seeded script `make_beans.py` next to it).

> **Task.**
> The file `beans.csv` has columns `species` (label:
> `alba`/`brio`/`cava`), `length_mm`, `width_mm`, `mass_g`,
> `moisture_pct`.
> Write code that:
> **(a)** loads the file into `beans` and reports `n_rows`;
> **(b)** computes `class_counts` — rows per species;
> **(c)** computes `n_heavy` — the number of beans with
> `mass_g > 0.7` — and `heavy_cava_frac`, the fraction of those heavy
> beans whose species is `cava`;
> **(d)** builds `X` from the four measurement columns *in the order
> listed above* and `y` from `species`, with `X.shape == (n_rows, 4)`
> and `X.dtype` float.
> **Constraints: no loops (zero points), no manual file parsing —
> `pd.read_csv` only.**

**Solution, narrated.**
(a) is the round trip of Section 2; (b) is `value_counts`; (c) is two
masks — note the *fraction within the heavy subset* means the mask mean
is taken **after** filtering; (d) is the Section 6 bridge, ending with
the shape ritual.

In [ ]:
import os
from pathlib import Path
_env_root = os.environ.get("USAAIO_BOOK_ROOT")
if _env_root:
    book_root = Path(_env_root).resolve()
else:
    _start = Path.cwd().resolve()
    book_root = next(p for p in [_start, *_start.parents] if (p / "syllabus.md").is_file() and (p / "curriculum").is_dir())
DATA_DIR = book_root / "units" / "C4-classical-ml-practice" / "practice" / "data"

# (a) load + row count
beans = pd.read_csv(DATA_DIR / "beans.csv")
n_rows = beans.shape[0]
print("n_rows:", n_rows)

# (b) class balance
class_counts = beans["species"].value_counts()
print(class_counts)

# (c) heavy beans, and cava's share of them
heavy_mask = beans["mass_g"] > 0.7
n_heavy = int(heavy_mask.sum())
heavy_cava_frac = (beans.loc[heavy_mask, "species"] == "cava").mean()
print("n_heavy:", n_heavy, "| heavy_cava_frac:", round(heavy_cava_frac, 4))

# (d) the NumPy bridge
FEATURES = ["length_mm", "width_mm", "mass_g", "moisture_pct"]
X = beans[FEATURES].to_numpy()
y = beans["species"].to_numpy()
print("X:", X.shape, X.dtype, "| y:", y.shape)
assert X.shape == (n_rows, 4) and y.shape == (n_rows,)

Ninety beans, thirty per class, 33 heavy ones of which 28 are `cava`
(fraction ≈ 0.85) — and the `(X, y)` pair ready for Session 2's
classifier.
Grader's-eye view: every deliverable is a *named identifier with a
pinned shape or type*; a correct analysis stored under the wrong name,
or an `X` with columns in a different order, scores zero on the real
paper's register.

### Checkpoint 7

1. Extend the task: compute `light_frac` — the fraction of *all* beans
   with `mass_g < 0.5` — and `n_dry_alba` — the number of `alba` beans
   with `moisture_pct` below 10.5.
2. Redo (d) with only `length_mm` and `moisture_pct` as features (that
   order).
   What must `X.shape` become, and what does row 0 of `X` contain?
3. Why is `(beans.loc[heavy_mask, "species"] == "cava").mean()` NOT
   the same number as
   `((beans["mass_g"] > 0.7) & (beans["species"] == "cava")).mean()`?
   What does each denominator count?

## 8. Common Pitfalls I

**Pitfall 1 — the unparenthesized mask.**
`&` binds *tighter* than comparisons, so
`df["mass_g"] > 0.7 & (df["species"] == "cava")` computes
`0.7 & (...)` first — elementwise-anding a float with booleans — and
either crashes or silently compares against garbage:

In [ ]:
try:
    bad = beans[beans["mass_g"] > 0.7 & (beans["species"] == "cava")]   # BROKEN
    print("no crash, but rows selected:", bad.shape[0], " <- wrong question answered")
except TypeError as err:
    print("TypeError --", err)

good = beans[(beans["mass_g"] > 0.7) & (beans["species"] == "cava")]    # parentheses
print("correct row count:", good.shape[0])                              # 28

Habit: in a combined mask, *every* comparison gets its own parentheses,
no exceptions.

**Pitfall 2 — Series where a matrix was needed (and vice versa).**
`df["mass_g"]` is 1-D; `df[["mass_g"]]` is 2-D with one column.
After `.to_numpy()` they become `(n,)` and `(n, 1)` — and shape-contract
graders (and broadcasting) treat those very differently:

In [ ]:
a = beans["mass_g"].to_numpy()       # (90,)   -- a vector
b = beans[["mass_g"]].to_numpy()     # (90, 1) -- a one-column matrix
print("single brackets:", a.shape, "| double brackets:", b.shape)

# A task demanding X.shape == (n, 1) fails with (n,), and vice versa:
print("(90,) == (90, 1)?", a.shape == b.shape)

Fix: decide up front whether the contract wants a vector or a matrix,
and pick the bracket form to match — then verify with the shape ritual.

**Pitfall 3 — the label column leaks into `X`.**
Including `species` among the features does not crash `to_numpy()`; it
quietly produces an **`object`-dtype array** in which every number has
been demoted to a Python object.
Arithmetic then breaks — or worse, a distance computation "works" by
comparing strings:

In [ ]:
X_leaky = beans[["species", "length_mm", "mass_g"]].to_numpy()   # BROKEN
print("dtype:", X_leaky.dtype)               # object, not float
try:
    X_leaky.mean(axis=0)
except TypeError as err:
    print("TypeError --", err)

X_clean = beans[["length_mm", "mass_g"]].to_numpy()
print("clean dtype:", X_clean.dtype, "| means:", X_clean.mean(axis=0).round(3))

The shape ritual catches this instantly: `X.dtype` must be float.
(There is also a modeling version of this leak — a feature that *is*
the label in disguise gives a perfect-looking classifier that knows
nothing.)

**Pitfall 4 — `iloc` after filtering.**
Filtered frames keep their original index labels, so *position* and
*label* stop agreeing.
`heavy.iloc[0]` is the first heavy row; `heavy.loc[0]` asks for the row
whose **label** is 0 — which may not be heavy at all, or may be missing
entirely:

In [ ]:
heavy = beans[beans["mass_g"] > 0.7]
print("heavy index head:", list(heavy.index[:5]))    # original labels survive
print("first heavy row by POSITION (iloc): mass =", heavy.iloc[0]["mass_g"])
row0 = heavy.loc[heavy.index[0]]                     # correct label-based access
print("same row by its LABEL:", heavy.index[0], "-> mass =", row0["mass_g"])
try:
    heavy.loc[0]                                     # label 0 was filtered out
except KeyError as err:
    print("KeyError -- label", err, "is not in the filtered frame")

Fix: after filtering, use `.iloc` for "first/last/…-th row", `.loc`
only with labels you *know* survived, and `.reset_index(drop=True)`
when downstream code assumes 0..n-1 labels.

### Checkpoint 8

1. A classmate writes
   `beans[beans["length_mm"] > 14 & (beans["mass_g"] > 0.7)]` and gets
   a `TypeError`.
   Which pitfall is this, and what is the corrected line?
2. A task requires `X.shape == (90, 1)` holding only `mass_g`.
   Give the one-line bridge that satisfies it, and the near-miss line
   that produces `(90,)` instead.
3. Explain in one sentence why `X.dtype == object` after the bridge is
   *always* a red flag, and name the most common cause.

## Checkpoint Answers

<details><summary><b>Checkpoint 1</b></summary>

1. `pets = pd.DataFrame({"name": ["Ada", "Bo", "Cy"], "weight_kg": [4.2, 11.0, 7.5]})`
   → shape `(3, 2)`; dtypes `object`/`str` for `name`, `float64` for
   `weight_kg`.
2. A DataFrame stores each column as its own Series with its own
   dtype, while one ndarray has a single dtype shared by every entry.
3. A **Series** — single-name brackets always return the 1-D labeled
   column.

</details>

<details><summary><b>Checkpoint 2</b></summary>

1. An unnamed first column (loaded as `Unnamed: 0`) holding the old
   row labels 0–4 — written because `to_csv` was called without
   `index=False`.
2. A non-numeric cell somewhere in the column — a typo like `0,42`, a
   unit like `"12 g"`, or a placeholder like `"missing"` — forces the
   whole column to `object`; `df.dtypes` catches it.
3. `df.shape` (rows); `df.describe()` (mins/maxes expose absurd
   values).

</details>

<details><summary><b>Checkpoint 3</b></summary>

1. `mini["length_mm"].max()` → 16.0;
   `mini[["mass_g", "length_mm"]].shape` → `(5, 2)`.
2. `(5,)` versus `(5, 1)` — Series (1-D) versus one-column DataFrame
   (2-D).
3. `mini["species"].value_counts()` → alba 2, cava 2, brio 1.

</details>

<details><summary><b>Checkpoint 4</b></summary>

1. `((mini["length_mm"] < 12) | (mini["mass_g"] > 0.8)).sum()` → 3
   (the two short albas and the 0.85 g cava).
2. `(mini["species"] == "cava").mean()` → 0.4.
3. `&` binds tighter than `>`, so Python first evaluates
   `0.6 & (mini["species"] == "cava")` — anding a float with a boolean
   Series — then compares `mass_g` with that result: the wrong
   computation even when it happens not to crash.

</details>

<details><summary><b>Checkpoint 5</b></summary>

1. `mini.groupby("species")["length_mm"].mean()` → alba 11.05,
   brio 13.10, cava 15.90: `cava` is longest.
2. If the classes have (nearly) the same distribution of a feature,
   knowing that feature's value says (nearly) nothing about the class
   — distances along it are noise.
3. `mini.groupby("species").size()`; same answer as
   `mini["species"].value_counts()`.

</details>

<details><summary><b>Checkpoint 6</b></summary>

1. `X2 = mini[["mass_g", "length_mm"]].to_numpy()` → `(5, 2)` float;
   `y2 = mini["species"].to_numpy()` → `(5,)`.
   `X2[:, 0]` is **mass** — column 0 is whatever the list put first.
2. The list is the single source of truth for column order; writing
   names inline twice lets the two copies drift, silently permuting
   `X`'s columns relative to what later code assumes.
3. `X2` must be numeric (float64 here); `y2` may be text — labels are
   names, not quantities.

</details>

<details><summary><b>Checkpoint 7</b></summary>

1. `light_frac = (beans["mass_g"] < 0.5).mean()`;
   `n_dry_alba = ((beans["species"] == "alba") & (beans["moisture_pct"] < 10.5)).sum()`.
2. `X.shape == (90, 2)`; row 0 holds bean 0's `length_mm` and
   `moisture_pct`, in that order.
3. The first divides by the number of **heavy** beans (33) — a
   fraction *within* the heavy subset; the second divides by **all**
   beans (90).
   Same numerator (28), different denominators: ≈ 0.85 vs ≈ 0.31.

</details>

<details><summary><b>Checkpoint 8</b></summary>

1. Pitfall 1 (mask precedence).
   `beans[(beans["length_mm"] > 14) & (beans["mass_g"] > 0.7)]`.
2. `X = beans[["mass_g"]].to_numpy()` → `(90, 1)`;
   the near-miss `beans["mass_g"].to_numpy()` gives `(90,)`.
3. An object-dtype `X` means at least one non-numeric value entered
   the matrix, so no arithmetic (means, distances, scaling) is
   trustworthy; the usual cause is the label (or another text column)
   included in the feature list.

</details>